In [1]:
from decouple import config
from sqlalchemy import create_engine
import pandas as pd
from datetime import datetime, timedelta
import time 
from datetime import datetime
from sqlalchemy import text
import oracledb
import sys

In [2]:
oracledb.init_oracle_client()
oracledb.version = "8.3.0"
sys.modules["cx_Oracle"] = oracledb
un = config("USER4_BDI_POSTGRES")
pw = config("PASS4_BDI_POSTGRES")
hostname=config("HOST4_BDI_POSTGRES")
service_name="WNET"
port = 1527

engine0 = create_engine(f'oracle://{un}:{pw}@',connect_args={
        "host": hostname,
        "port": port,
        "service_name": service_name
    }
)

connection0 = engine0.connect()

In [3]:
DB_USER='root'
DB_PASSWORD='OJlehXTRlEa6^zsFNILjnVoew9ku=E'
DB_NAME='essidb'
DB_PORT="5432"
DB_HOST='read-only-powerbi.cktgjnvajmd8.us-east-1.rds.amazonaws.com'
cadena1  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine1 = create_engine(cadena1)
connection1 = engine1.connect()

In [4]:
DB_USER='postgres'
DB_PASSWORD='AKindOfMagic'
DB_NAME="dw_essalud"
DB_PORT="5432"
DB_HOST='10.0.1.228'
cadena2  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine2 = create_engine(cadena2)
connection2 = engine2.connect()

In [5]:
DB_USER='postgres'
DB_PASSWORD='AKindOfMagic'
DB_NAME="indicadores_gctic"
DB_PORT="5432"
DB_HOST='10.0.1.228'
cadena3  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine3 = create_engine(cadena3)
connection3 = engine3.connect()

In [6]:
DB_USER='postgres'
DB_PASSWORD='zxcvqwer159A-'
DB_NAME="indicadores_gctic"
DB_PORT="5432"
DB_HOST='10.0.28.15'
cadena4  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine4 = create_engine(cadena4)
connection4 = engine4.connect()

In [7]:
citas_medicas_query="""SELECT
    ras.REDASISCOD,
    c.ESTCITCOD,
    c.CITAMBPROCONFEC AS FECHA,
    COUNT(*) AS CANTIDAD
FROM 
    CTCAM10 c
LEFT OUTER JOIN 
    CMCAS10 cas ON c.CITAMBCENASICOD = cas.CENASICOD
LEFT OUTER JOIN 
    CMRAS10 ras ON cas.REDASISCOD = ras.REDASISCOD
WHERE 
    c.CITAMBPROCONFEC >= TO_DATE('2023-01-01', 'YYYY-MM-DD')
    AND c.MODOTORCITACOD = '6'
GROUP BY 
    ras.REDASISCOD, c.CITAMBCENASICOD, c.ESTCITCOD, c.CITAMBPROCONFEC
"""
citas_medicas = pd.read_sql_query(citas_medicas_query, con=connection0)

In [8]:
fecha_ultimo_mes_cerrado = pd.to_datetime(pd.to_datetime('today').strftime('%Y-%m-%d')).to_period('M').to_timestamp('M') + pd.offsets.MonthEnd(0)

In [9]:
red = pd.read_sql_query(f"SELECT id_red,cod_red FROM dim_red", con=connection2)

citas_medicas=pd.merge(citas_medicas,red,how='left', left_on='redasiscod', right_on='cod_red')
citas_medicas=citas_medicas.drop('redasiscod',axis=1)
citas_medicas=citas_medicas.drop('cod_red',axis=1)

In [10]:
citas_medicas_filtradas = citas_medicas[(citas_medicas['fecha'] <= fecha_ultimo_mes_cerrado)]

In [11]:
# Convertimos la columna FECHA a tipo fecha
citas_medicas_filtradas['fecha'] = pd.to_datetime(citas_medicas_filtradas['fecha'])

# Creamos el DataFrame para citas_medicas_solicitadas
citas_medicas_solicitadas = citas_medicas_filtradas.groupby([
    citas_medicas_filtradas['id_red'], 
    citas_medicas_filtradas['fecha'].dt.year.rename('año'),  # Renombramos la columna
    citas_medicas_filtradas['fecha'].dt.month.rename('mes')  # Renombramos la columna
]).agg({'cantidad': 'sum'}).reset_index()

# Creamos el DataFrame para citas_medicas_atendidas
citas_medicas_atendidas = citas_medicas_filtradas[citas_medicas_filtradas['estcitcod'] == '4'].groupby([
    citas_medicas_filtradas['id_red'], 
    citas_medicas_filtradas['fecha'].dt.year.rename('año'),  # Renombramos la columna
    citas_medicas_filtradas['fecha'].dt.month.rename('mes')  # Renombramos la columna
]).agg({'cantidad': 'sum'}).reset_index()


/tmp/ipykernel_714527/1396901530.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  citas_medicas_filtradas['fecha'] = pd.to_datetime(citas_medicas_filtradas['fecha'])


In [12]:
citas_medicas_solicitadas.to_sql(name=f'citas_medicas_solicitadas', con=connection4, if_exists='replace', index=False,chunksize=100)
citas_medicas_atendidas.to_sql(name=f'citas_medicas_atendidas', con=connection4, if_exists='replace', index=False,chunksize=100)

397

In [13]:
usuarios_miconsulta_query="""SELECT
  EXTRACT(YEAR FROM u.date_create) AS año,
  EXTRACT(MONTH FROM u.date_create) AS mes,
  c.cod_red,
  SUM(COUNT(*)) OVER (PARTITION BY c.cod_red ORDER BY EXTRACT(YEAR FROM u.date_create), EXTRACT(MONTH FROM u.date_create)) AS cantidad_acumulada
FROM
  usuario u
LEFT JOIN paciente p ON u.id_usuario = p.id_paciente
LEFT JOIN persona per ON u.id_usuario = per.id_persona
LEFT JOIN centro c ON c.cen_asi_cod = p.cod_centro
LEFT JOIN red r ON r.cod_red = c.cod_red
WHERE
  TO_DATE(per.fecha_nacimiento, 'DD/MM/YYYY') <= (CURRENT_DATE - INTERVAL '18 years')
GROUP BY
  EXTRACT(YEAR FROM u.date_create),
  EXTRACT(MONTH FROM u.date_create),
  c.cod_red
ORDER BY
  EXTRACT(YEAR FROM u.date_create),
  EXTRACT(MONTH FROM u.date_create),
  c.cod_red
"""
usuarios_miconsulta = pd.read_sql_query(usuarios_miconsulta_query, con=connection1)


In [14]:
usuarios_miconsulta = usuarios_miconsulta[usuarios_miconsulta['año'] == 2024].reset_index()

In [15]:
red = pd.read_sql_query(f"SELECT id_red,cod_red FROM dim_red", con=connection2)

usuarios_miconsulta=pd.merge(usuarios_miconsulta,red,how='left', left_on='cod_red', right_on='cod_red')
usuarios_miconsulta=usuarios_miconsulta.drop('cod_red',axis=1)
usuarios_miconsulta=usuarios_miconsulta.drop('index',axis=1)

In [16]:
usuarios_miconsulta

,año,mes,cantidad_acumulada,id_red
0,2024.0,1.0,89550.0,1.0
1,2024.0,1.0,86603.0,2.0
2,2024.0,1.0,105664.0,3.0
3,2024.0,1.0,1425.0,4.0
4,2024.0,1.0,15257.0,5.0
...,...,...,...,...
251,2024.0,8.0,2477.0,27.0
252,2024.0,8.0,1420.0,28.0
253,2024.0,8.0,2255.0,34.0
254,2024.0,8.0,251.0,31.0


In [17]:
# Filtrar las filas donde el año sea 2024 y el mes sea 4
filtro = (usuarios_miconsulta['año'] == 2024) & (usuarios_miconsulta['mes'] == 5)
usuarios_miconsulta_filtrado = usuarios_miconsulta.loc[filtro]

# Sumar la columna 'cantidad_acumulada' del DataFrame filtrado
total_cantidad_acumulada = usuarios_miconsulta_filtrado['cantidad_acumulada'].sum()

print("La cantidad acumulada total para el año 2024 y mes 5 es:", total_cantidad_acumulada)

La cantidad acumulada total para el año 2024 y mes 5 es: 604882.0


In [18]:
# poblacion_asegurada_red_query = """
# SELECT ras.REDASISCOD, COUNT(*) as cantidad
# FROM CMPAC10 pac
# LEFT OUTER JOIN CMPER10 per ON pac.PACSECNUM = per.PERSECNUM
# LEFT OUTER JOIN CMCAS10 cas ON cas.CENASICOD = pac.CENASICOD
# LEFT OUTER JOIN CMRAS10 ras ON cas.REDASISCOD = ras.REDASISCOD
# WHERE pac.TIPOPACICOD IN ('1', '2', 'c')
# AND pac.ESTHISCLICOD = '1'
# AND pac.PACESTREGCOD = '1'
# AND MONTHS_BETWEEN(SYSDATE, per.PERNACFEC)/12 > 18
# GROUP BY ras.REDASISCOD
# """

# poblacion_asegurada_red = pd.read_sql_query(poblacion_asegurada_red_query, con=connection0)

In [19]:
# red = pd.read_sql_query(f"SELECT id_red,cod_red FROM dim_red", con=connection2)

# poblacion_asegurada_red=pd.merge(poblacion_asegurada_red,red,how='left', left_on='redasiscod', right_on='cod_red')
# poblacion_asegurada_red=poblacion_asegurada_red.drop('redasiscod',axis=1)
# poblacion_asegurada_red=poblacion_asegurada_red.drop('cod_red',axis=1)

In [20]:
total_poblacion_asegurada = 9479841

In [21]:
usuarios_miconsulta_18 = usuarios_miconsulta
usuarios_miconsulta_18['porcentaje'] = (usuarios_miconsulta_18['cantidad_acumulada'] / total_poblacion_asegurada) * 100
usuarios_miconsulta_18 = usuarios_miconsulta_18.drop('cantidad_acumulada',axis=1)

In [22]:
usuarios_miconsulta_18

,año,mes,id_red,porcentaje
0,2024.0,1.0,1.0,0.944636
1,2024.0,1.0,2.0,0.913549
2,2024.0,1.0,3.0,1.114618
3,2024.0,1.0,4.0,0.015032
4,2024.0,1.0,5.0,0.160942
...,...,...,...,...
251,2024.0,8.0,27.0,0.026129
252,2024.0,8.0,28.0,0.014979
253,2024.0,8.0,34.0,0.023787
254,2024.0,8.0,31.0,0.002648


In [23]:
usuarios_miconsulta.to_sql(name=f'usuarios_miconsulta', con=connection3, if_exists='replace', index=False,chunksize=10000)

256

In [24]:
usuarios_solcita_query="""SELECT
  EXTRACT(YEAR FROM date_create) AS año,
  EXTRACT(MONTH FROM date_create) AS mes,
  COUNT(DISTINCT numero_doc_ident) AS cantidad_personas,
  cen_asi_cod AS centro
FROM
  cita
WHERE
  EXTRACT(YEAR FROM date_create) >= 2024
GROUP BY
  EXTRACT(YEAR FROM date_create),
  EXTRACT(MONTH FROM date_create),
  cen_asi_cod
ORDER BY
  EXTRACT(YEAR FROM date_create),
  EXTRACT(MONTH FROM date_create),
  cen_asi_cod
"""

usuarios_solcita = pd.read_sql_query(usuarios_solcita_query, con=connection1)


In [25]:
cas = pd.read_sql_query(f"SELECT id_cas,cod_cas, cod_red FROM dim_cas", con=connection2)
red = pd.read_sql_query(f"SELECT id_red,cod_red FROM dim_red", con=connection2)

In [26]:
usuarios_solcita=pd.merge(usuarios_solcita,cas,how='left', left_on='centro', right_on='cod_cas')
usuarios_solcita=pd.merge(usuarios_solcita,red,how='left', left_on='cod_red', right_on='cod_red')
usuarios_solcita=usuarios_solcita.drop('cod_red',axis=1)
usuarios_solcita=usuarios_solcita.drop('cod_cas',axis=1)
usuarios_solcita=usuarios_solcita.drop('id_cas',axis=1)

In [27]:
# Creamos el DataFrame para citas_medicas_solicitadas
usuarios_solcita = usuarios_solcita.groupby([
    usuarios_solcita['id_red'],
    usuarios_solcita['año'],
    usuarios_solcita['mes']
]).agg({'cantidad_personas': 'sum'}).reset_index()

In [28]:
usuarios_solcita.tail(30)

,id_red,año,mes,cantidad_personas
233,30,2024.0,2.0,1
234,30,2024.0,3.0,1
235,30,2024.0,4.0,1
236,30,2024.0,5.0,2
237,30,2024.0,6.0,1
238,30,2024.0,7.0,3
239,31,2024.0,1.0,359
240,31,2024.0,2.0,335
241,31,2024.0,3.0,407
242,31,2024.0,4.0,479


In [29]:
usuarios_solcita_mi_consulta = pd.merge(usuarios_miconsulta, usuarios_solcita, on=['id_red', 'año', 'mes'], how='left')

# Calculamos el porcentaje
usuarios_solcita_mi_consulta['porcentaje'] = ( usuarios_solcita_mi_consulta['cantidad_personas'] / usuarios_solcita_mi_consulta['cantidad_acumulada'] ) * 100

# Seleccionamos las columnas requeridas
usuarios_solcita_mi_consulta = usuarios_solcita_mi_consulta[['id_red', 'año', 'mes','cantidad_personas','cantidad_acumulada', 'porcentaje',]]

In [30]:
usuarios_solcita_mi_consulta = usuarios_solcita_mi_consulta[
    (~pd.isnull(usuarios_solcita_mi_consulta['id_red'])) &
    (~usuarios_solcita_mi_consulta['id_red'].isin([29, 30, 31, 32]))
].reset_index(drop=True)

In [31]:
usuarios_solcita_mi_consulta

,id_red,año,mes,cantidad_personas,cantidad_acumulada,porcentaje
0,1.0,2024.0,1.0,3304.0,89550.0,3.689559
1,2.0,2024.0,1.0,5313.0,86603.0,6.134891
2,3.0,2024.0,1.0,7583.0,105664.0,7.176522
3,4.0,2024.0,1.0,34.0,1425.0,2.385965
4,5.0,2024.0,1.0,305.0,15257.0,1.999082
...,...,...,...,...,...,...
235,25.0,2024.0,8.0,16.0,4202.0,0.380771
236,26.0,2024.0,8.0,49.0,6520.0,0.751534
237,27.0,2024.0,8.0,32.0,2477.0,1.291885
238,28.0,2024.0,8.0,18.0,1420.0,1.267606


In [32]:
usuarios_solcita_mi_consulta.to_sql(name=f'usuarios_solcita_mi_consulta', con=connection4, if_exists='replace', index=False,chunksize=10000)

240

In [33]:
pob_asegurada_red = pd.read_excel('Libro2.xlsx')

In [34]:
usuarios_miconsulta

,año,mes,cantidad_acumulada,id_red,porcentaje
0,2024.0,1.0,89550.0,1.0,0.944636
1,2024.0,1.0,86603.0,2.0,0.913549
2,2024.0,1.0,105664.0,3.0,1.114618
3,2024.0,1.0,1425.0,4.0,0.015032
4,2024.0,1.0,15257.0,5.0,0.160942
...,...,...,...,...,...
251,2024.0,8.0,2477.0,27.0,0.026129
252,2024.0,8.0,1420.0,28.0,0.014979
253,2024.0,8.0,2255.0,34.0,0.023787
254,2024.0,8.0,251.0,31.0,0.002648


In [35]:
pob_asegurada_red

,id_red,cod_red,des_red,cantidad
0,1,5,RED PRESTACIONAL SABOGAL ...,1462478
1,2,6,RED PRESTACIONAL ALMENARA ...,1283801
2,3,7,RED PRESTACIONAL REBAGLIATI ...,1626360
3,4,8,RED ASISTENCIAL TUMBES ...,54211
4,5,9,RED ASISTENCIAL PIURA ...,542626
5,6,10,RED ASISTENCIAL LAMBAYEQUE ...,466670
6,7,12,RED ASISTENCIAL CAJAMARCA ...,156026
7,8,13,RED ASISTENCIAL AMAZONAS ...,63529
8,9,15,RED ASISTENCIAL LA LIBERTAD ...,541928
9,10,16,RED ASISTENCIAL ANCASH ...,184590


In [36]:
pob_asegurada_red['cantidad'].sum()

9479841

In [37]:
usuarios_miconsulta_18 = pd.merge(usuarios_miconsulta, pob_asegurada_red, on='id_red', how='inner')

# Calcular el porcentaje
usuarios_miconsulta_18['porcentaje'] = (usuarios_miconsulta_18['cantidad_acumulada'] / usuarios_miconsulta_18['cantidad']) * 100

# Seleccionar las columnas requeridas
usuarios_miconsulta_18 = usuarios_miconsulta_18[['id_red', 'año', 'mes','cantidad_acumulada','porcentaje']]


In [38]:
usuarios_miconsulta_18

,id_red,año,mes,cantidad_acumulada,porcentaje
0,1.0,2024.0,1.0,89550.0,6.123169
1,1.0,2024.0,2.0,93757.0,6.410831
2,1.0,2024.0,3.0,98909.0,6.763110
3,1.0,2024.0,4.0,104714.0,7.160039
4,1.0,2024.0,5.0,110089.0,7.527566
...,...,...,...,...,...
235,34.0,2024.0,4.0,1731.0,3.537996
236,34.0,2024.0,5.0,1932.0,3.948821
237,34.0,2024.0,6.0,2087.0,4.265626
238,34.0,2024.0,7.0,2231.0,4.559948


In [39]:
usuarios_miconsulta_18.to_sql(name=f'usuarios_miconsulta_18', con=connection4, if_exists='replace', index=False,chunksize=10000)

240

In [40]:
connection1.close()
connection2.close()
engine1.dispose()
engine2.dispose()